# Tutor Inteligente — OCR Espanol + Detector YOLO
**Kaggle Dual T4 GPU | Marzo 2026**

Pipeline completo:
1. Construir `char_map.json` desde `yolo_dataset_final`
2. Extraer crops del dataset YOLO para el Clasificador (EfficientNet-B2)
3. Entrenar Clasificador con DataParallel 2xT4, AMP FP16, MLflow
4. Exportar Clasificador a ONNX opset 17 + INT8
5. Entrenar Detector YOLOv8n con MLflow (mismo tracking URI)
6. Exportar Detector a ONNX 640x640
7. Copiar artefactos + paquete ZIP final

**Dataset:** `yolo_dataset_final` subido a Kaggle
**MLflow:** ambos entrenamientos en el mismo `./mlruns`


In [ ]:
!pip install timm
!pip install onnx
!pip install onnxruntime-gpu
!pip install onnxruntime-tools
!pip install onnxsim
!pip install albumentations
!pip install tqdm
!pip install scikit-learn
!pip install matplotlib
!pip install seaborn
!pip install onnxconverter-common
!pip install mlflow
!pip install ultralytics
!pip install pillow-avif
!pip install yalm

In [ ]:
import torch
import timm
import onnx
import onnxruntime
import mlflow
import ultralytics

print(f'PyTorch    : {torch.__version__}')
print(f'TIMM       : {timm.__version__}')
print(f'ONNX       : {onnx.__version__}')
print(f'ORT        : {onnxruntime.__version__}')
print(f'MLflow     : {mlflow.__version__}')
print(f'Ultralytics: {ultralytics.__version__}')
print(f'CUDA       : {torch.version.cuda}')

In [ ]:
import os, random, time, json, shutil, re
import numpy as np
import torch
import torch.nn as nn
from pathlib import Path
from collections import Counter

SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

NUM_GPUS = torch.cuda.device_count()
DEVICE   = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Dispositivo : {DEVICE}  |  GPUs: {NUM_GPUS}')
for i in range(NUM_GPUS):
    p = torch.cuda.get_device_properties(i)
    print(f'  GPU {i}: {p.name}  |  VRAM: {p.total_memory/1e9:.1f} GB')

torch.backends.cudnn.benchmark     = True
torch.backends.cudnn.deterministic = False
torch.backends.cuda.matmul.allow_tf32 = True

# ---- Localizar dataset en /kaggle/input/ ----
_DATASET_ROOT = None

# Buscamos en todo /kaggle/input el archivo dataset.yaml para encontrar la carpeta raíz
paths_encontrados = list(Path('/kaggle/input').rglob('dataset.yaml'))

if paths_encontrados:
    # Si lo encuentra, la raíz es la carpeta que contiene al yaml
    _DATASET_ROOT = paths_encontrados[0].parent
    print(f"✅ Dataset encontrado en: {_DATASET_ROOT}")
else:
    # Fallback si lo subiste manualmente a /working
    fallback = Path('/kaggle/working/yolo_dataset_final')
    if fallback.exists():
        _DATASET_ROOT = fallback
        print(f"⚠️ Usando fallback en: {_DATASET_ROOT}")

# Verificación final
if _DATASET_ROOT is None or not _DATASET_ROOT.exists():
    # Listamos que hay en input para ayudarte a depurar
    print("\n❌ ERROR: No se encontró 'dataset.yaml'. Contenido de /kaggle/input:")
    for p in Path('/kaggle/input').iterdir():
        print(f" - {p.name}")
    raise AssertionError("Dataset no encontrado. Revisa que el dataset esté cargado correctamente en 'Add Data'.")

DATASET_YAML = str(_DATASET_ROOT / 'dataset.yaml')
IMAGES_TRAIN = _DATASET_ROOT / 'images' / 'train'
IMAGES_VAL   = _DATASET_ROOT / 'images' / 'val'
LABELS_TRAIN = _DATASET_ROOT / 'labels' / 'train'
LABELS_VAL   = _DATASET_ROOT / 'labels' / 'val'

OUTPUT_DIR = Path('/kaggle/working')
CKPT_DIR   = OUTPUT_DIR / 'checkpoints'
ONNX_DIR   = OUTPUT_DIR / 'onnx'
YOLO_DIR   = OUTPUT_DIR / 'runs' / 'detect'
MLRUNS_DIR = str(OUTPUT_DIR / 'mlruns')

for d in [CKPT_DIR, ONNX_DIR, YOLO_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# ---- Config clasificador ----
CFG = dict(
    img_size=64, num_workers=2, pin_memory=True,
    prefetch_factor=2, persistent_workers=False,
    model_name='efficientnet_b2', pretrained=True, drop_rate=0.3,
    epochs=60, batch_size=256, lr=3e-3, weight_decay=1e-4,
    warmup_epochs=5, grad_clip=1.0, label_smoothing=0.1,
    use_amp=True, mixup_alpha=0.4, cutmix_alpha=1.0, patience=12,
    ckpt_dir=str(CKPT_DIR), onnx_dir=str(ONNX_DIR),
    mlflow_uri=MLRUNS_DIR, mlflow_experiment='tutor_inteligente_ocr',
)

# ---- Config detector YOLO ----
YOLO_CFG = dict(
    model_variant='yolov8n.pt', epochs=80, batch=32,
    device='0,1' if NUM_GPUS >= 2 else ('0' if NUM_GPUS == 1 else 'cpu'),
    workers=8, img_size=640, lr0=0.02, lrf=0.01, momentum=0.937,
    weight_decay=5e-4, warmup_epochs=5.0, amp=True, cache='ram',
    patience=20, project=str(YOLO_DIR), name='char_detector_t4',
)

print(f'\nBatch clasificador efectivo : {CFG["batch_size"] * max(NUM_GPUS,1)}')
print(f'Dispositivo YOLO            : {YOLO_CFG["device"]}')
# ---- Localizar dataset de forma única ----
import os
from pathlib import Path

# Buscamos el archivo yaml en la carpeta de entrada (Kaggle Input)
paths_yaml = list(Path('/kaggle/input').rglob('dataset.yaml'))

if paths_yaml:
    _DATASET_ROOT = paths_yaml[0].parent
    print(f"✅ Dataset detectado en ruta real: {_DATASET_ROOT}")
else:
    # Si por alguna razón no está en input, intentamos en working
    _DATASET_ROOT = Path('/kaggle/working/yolo_dataset_final')
    print(f"⚠️ Buscando en fallback: {_DATASET_ROOT}")

# --- ESTA ES LA PARTE IMPORTANTE: No permitas que nada sobrescriba _DATASET_ROOT después de esto ---

assert _DATASET_ROOT.exists(), f"ERROR: La ruta {_DATASET_ROOT} no existe. Verifica 'Add Data'."

# Configurar rutas hijas usando la variable que YA encontramos
DATASET_YAML = str(_DATASET_ROOT / 'dataset.yaml')
IMAGES_TRAIN = _DATASET_ROOT / 'images' / 'train'
IMAGES_VAL   = _DATASET_ROOT / 'images' / 'val'
LABELS_TRAIN = _DATASET_ROOT / 'labels' / 'train'
LABELS_VAL   = _DATASET_ROOT / 'labels' / 'val'

print(f"Ruta final del YAML: {DATASET_YAML}")

OUTPUT_DIR = Path('/kaggle/working')
CKPT_DIR   = OUTPUT_DIR / 'checkpoints'
ONNX_DIR   = OUTPUT_DIR / 'onnx'
YOLO_DIR   = OUTPUT_DIR / 'runs' / 'detect'
MLRUNS_DIR = str(OUTPUT_DIR / 'mlruns')

for d in [CKPT_DIR, ONNX_DIR, YOLO_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# ---- Config clasificador ----
CFG = dict(
    img_size=64, num_workers=2, pin_memory=True,
    prefetch_factor=2, persistent_workers=False,
    model_name='efficientnet_b2', pretrained=True, drop_rate=0.3,
    epochs=60, batch_size=256, lr=3e-3, weight_decay=1e-4,
    warmup_epochs=5, grad_clip=1.0, label_smoothing=0.1,
    use_amp=True, mixup_alpha=0.4, cutmix_alpha=1.0, patience=12,
    ckpt_dir=str(CKPT_DIR), onnx_dir=str(ONNX_DIR),
    mlflow_uri=MLRUNS_DIR, mlflow_experiment='tutor_inteligente_ocr',
)

# ---- Config detector YOLO ----
YOLO_CFG = dict(
    model_variant='yolov8n.pt', epochs=80, batch=32,
    device='0,1' if NUM_GPUS >= 2 else ('0' if NUM_GPUS == 1 else 'cpu'),
    workers=8, img_size=640, lr0=0.02, lrf=0.01, momentum=0.937,
    weight_decay=5e-4, warmup_epochs=5.0, amp=True, cache='ram',
    patience=20, project=str(YOLO_DIR), name='char_detector_t4',
)

print(f'\nBatch clasificador efectivo : {CFG["batch_size"] * max(NUM_GPUS,1)}')
print(f'Dispositivo YOLO            : {YOLO_CFG["device"]}')


## Celda 3 — Construir char_map.json desde filenames del dataset

In [ ]:
# =============================================================================
# CELDA 3 - Construir char_map.json con las 107 clases reales
#
# FUENTE PRIMARIA: dataset_classes_report.json subido a Kaggle.
# Contiene la lista canonica de todas las clases objetivo (107).
# =============================================================================

EMNIST_CHARS = list('0123456789ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz')
PRIM_SLUG_MAP = {
    'linea_vertical'         : 'linea_vertical',
    'linea_horizontal'       : 'linea_horizontal',
    'linea_oblicua_derecha'  : 'linea_oblicua_derecha',
    'linea_oblicua_izquierda': 'linea_oblicua_izquierda',
    'curva'                  : 'curva',
    'circulo'                : 'circulo',
}

# ---- 1. Localizar dataset_classes_report.json en /kaggle/input/ ----
_REPORT_PATH = None
for candidate in Path('/kaggle/input').rglob('dataset_classes_report.json'):
    _REPORT_PATH = candidate
    break
if _REPORT_PATH is None:
    _fallback = Path('/kaggle/working/dataset_classes_report.json')
    if _fallback.exists():
        _REPORT_PATH = _fallback
print(f'dataset_classes_report.json : {_REPORT_PATH}')

# ---- 2. Cargar clases objetivo desde el reporte ----
if _REPORT_PATH and _REPORT_PATH.exists():
    with open(_REPORT_PATH, 'r', encoding='utf-8') as f:
        _report = json.load(f)
    target_classes  = _report.get('target_classes', [])
    _global_missing = set(_report.get('global_missing', []))
    _coverage       = _report.get('coverage_summary', {})
    print(f'Reporte: {len(target_classes)} clases objetivo | faltantes={len(_global_missing)} | cobertura={_coverage.get("coverage_pct","?")}')
else:
    print('[WARN] dataset_classes_report.json no encontrado. Usando EMNIST+primitivos como fallback.')
    target_classes  = EMNIST_CHARS + list(PRIM_SLUG_MAP.values())
    _global_missing = set(PRIM_SLUG_MAP.values())

# ---- 3. Contar imagenes decodificables por clase desde filenames ----
# Solo los archivos sinteticos cls*/prim*/char_* tienen la clase en el nombre.
# Los archivos de Spanish/HWC reales NO son decodificables (ver nota arriba).
def decode_char_from_stem(stem):
    m = re.match(r'^cls(\d{3})_', stem)
    if m:
        idx = int(m.group(1))
        return (EMNIST_CHARS[idx], True) if idx < len(EMNIST_CHARS) else (None, False)
    m = re.match(r'^prim_([a-z_]+)_', stem)
    if m and m.group(1) in PRIM_SLUG_MAP:
        return PRIM_SLUG_MAP[m.group(1)], True
    m = re.match(r'^char_(\d{2})_', stem)
    if m:
        idx = int(m.group(1))
        return (EMNIST_CHARS[idx], True) if idx < len(EMNIST_CHARS) else (None, False)
    m = re.match(r'^(.)_\d', stem)
    if m and m.group(1).strip() and len(m.group(1)) == 1:
        return m.group(1), False
    return None, False

print('Escaneando filenames en images/train ...')
char_freq_from_files = Counter()
all_stems = [p.stem for p in sorted(IMAGES_TRAIN.glob('*.jpg'))]
for stem in all_stems:
    char, _ = decode_char_from_stem(stem)
    if char:
        char_freq_from_files[char] += 1

total_imgs  = len(all_stems)
n_decodable = sum(char_freq_from_files.values())
print(f'  Total imagenes train     : {total_imgs:,}')
print(f'  Con clase decodificable  : {n_decodable:,}  ({n_decodable*100//max(total_imgs,1)}%)')
print(f'  Sin clase decodificable  : {total_imgs - n_decodable:,}  (Spanish/HWC renombrados)')
print(f'  Clases en filenames      : {len(char_freq_from_files)}')

# ---- 4. Construir char_map unificado con las 107 clases del reporte ----
seen = set(); ordered_chars = []
for c in target_classes:
    if c not in seen:
        seen.add(c); ordered_chars.append(c)

CHAR2IDX    = {c: i for i, c in enumerate(ordered_chars)}
IDX2CHAR    = {i: c for i, c in enumerate(ordered_chars)}
NUM_CLASSES = len(ordered_chars)

n_emnist = sum(1 for c in ordered_chars if c in set(EMNIST_CHARS))
n_prim   = sum(1 for c in ordered_chars if c in set(PRIM_SLUG_MAP.values()))
n_extra  = NUM_CLASSES - n_emnist - n_prim

classes_with_data    = [c for c in ordered_chars if char_freq_from_files.get(c, 0) > 0]
classes_without_data = [c for c in ordered_chars if char_freq_from_files.get(c, 0) == 0]

print(f'\nchar_map desde reporte:')
print(f'  NUM_CLASSES   : {NUM_CLASSES}')
print(f'  EMNIST        : {n_emnist}')
print(f'  Primitivos    : {n_prim}')
print(f'  Extras espanol: {n_extra}')
print(f'  Primeras 20   : {ordered_chars[:20]}')
print(f'  Ultimas  10   : {ordered_chars[-10:]}')
print(f'  Con crops decodificables : {len(classes_with_data)}/{NUM_CLASSES}')
print(f'  Sin crops decodificables : {len(classes_without_data)} -> no entrenaran el clasificador')
if classes_without_data:
    print(f'    Clases sin datos: {classes_without_data[:30]}')
    print('    FIX: re-ejecutar generate_synthetic_yolo.py para esas clases')
    print('    y preparar el dataset incluyendo esos sinteticos nuevos.')

# ---- 5. Guardar char_map.json ----
char_map_path = ONNX_DIR / 'char_map.json'
with open(char_map_path, 'w', encoding='utf-8') as f:
    json.dump({
        'idx2char'         : {str(i): c for i, c in IDX2CHAR.items()},
        'char2idx'         : CHAR2IDX,
        'num_classes'      : NUM_CLASSES,
        'class_freq'       : {c: char_freq_from_files.get(c, 0) for c in ordered_chars},
        'trainable_classes': len(classes_with_data),
        'source'           : str(_REPORT_PATH) if _REPORT_PATH else 'fallback',
    }, f, ensure_ascii=False, indent=2)
print(f'\nchar_map.json guardado -> {char_map_path}  ({NUM_CLASSES} clases)')


## Celda 4 — Dataset de crops para el clasificador

In [ ]:
import cv2
from PIL import Image

def extract_crop(img_bgr, label_line, img_size):
    parts = label_line.strip().split()
    if len(parts) < 5: return None
    xc, yc, w, h = float(parts[1]), float(parts[2]), float(parts[3]), float(parts[4])
    H, W = img_bgr.shape[:2]
    x1 = max(0, int((xc - w/2)*W)); y1 = max(0, int((yc - h/2)*H))
    x2 = min(W, int((xc + w/2)*W)); y2 = min(H, int((yc + h/2)*H))
    if x2 <= x1 or y2 <= y1: return None
    crop = cv2.cvtColor(img_bgr[y1:y2, x1:x2], cv2.COLOR_BGR2RGB)
    return cv2.resize(crop, (img_size, img_size), interpolation=cv2.INTER_CUBIC)

def build_samples(images_dir, labels_dir, split_name):
    samples, skip_cls, skip_lbl, skip_crop = [], 0, 0, 0
    img_paths = sorted(images_dir.glob('*.jpg'))
    print(f"  {split_name}: {len(img_paths):,} imagenes ...")
    for img_path in img_paths:
        char, _ = decode_char_from_stem(img_path.stem)
        if char is None or char not in CHAR2IDX:
            skip_cls += 1; continue
        class_idx = CHAR2IDX[char]
        lbl_path  = labels_dir / (img_path.stem + '.txt')
        if not lbl_path.exists():
            skip_lbl += 1; continue
        lines = lbl_path.read_text().strip().splitlines()
        if not lines:
            skip_lbl += 1; continue
        img_bgr = cv2.imread(str(img_path))
        if img_bgr is None:
            skip_crop += 1; continue
        for line in lines:
            crop = extract_crop(img_bgr, line, CFG['img_size'])
            if crop is not None:
                samples.append((crop, class_idx))
            else:
                skip_crop += 1
    print(f"    OK: {len(samples):,}  | skip_cls={skip_cls} skip_lbl={skip_lbl} skip_crop={skip_crop}")
    return samples

print("=== Construyendo crops del clasificador ===\n")
train_samples = build_samples(IMAGES_TRAIN, LABELS_TRAIN, 'train')
val_samples   = build_samples(IMAGES_VAL,   LABELS_VAL,   'val')

train_counts = Counter(s[1] for s in train_samples)
print(f"\nResumen:")
print(f"  Train crops    : {len(train_samples):,}")
print(f"  Val   crops    : {len(val_samples):,}")
print(f"  Clases en train: {len(train_counts)}/{NUM_CLASSES}")
missing_train = [IDX2CHAR[i] for i in range(NUM_CLASSES) if i not in train_counts]
if missing_train:
    print(f"  Clases sin datos: {missing_train[:20]}")


## Celda 5 — Augmentation pipeline

In [ ]:
import albumentations as A
from albumentations.pytorch import ToTensorV2

IMG_SIZE = CFG['img_size']

train_transform = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.OneOf([
        A.Perspective(scale=(0.02, 0.05), p=1.0),
        A.Affine(scale=(0.9,1.1), translate_percent={'x':(-0.05,0.05),'y':(-0.05,0.05)},
                 rotate=(-10,10), shear=(-5,5), p=1.0),
    ], p=0.7),
    A.OneOf([
        A.GaussNoise(std_range=(0.01,0.05), p=1.0),
        A.GaussianBlur(blur_limit=(3,5), p=1.0),
    ], p=0.4),
    A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.5),
    A.CoarseDropout(num_holes_range=(1,4), hole_height_range=(8,16),
                   hole_width_range=(8,16), fill=255, p=0.2),
    A.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]),
    ToTensorV2(),
])
val_transform = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]),
    ToTensorV2(),
])
print(f'Augmentation definido | img_size={IMG_SIZE}')


## Celda 6 — Dataset class y DataLoaders

In [ ]:
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from sklearn.model_selection import train_test_split

class CropDataset(Dataset):
    def __init__(self, samples, transform=None):
        self.samples = samples; self.transform = transform
    def __len__(self): return len(self.samples)
    def __getitem__(self, idx):
        img, label = self.samples[idx]
        if self.transform: img = self.transform(image=img)['image']
        return img, label

tr_smp, te_smp = train_test_split(
    train_samples, test_size=0.10, random_state=SEED,
    stratify=[s[1] for s in train_samples])

train_ds = CropDataset(tr_smp,      train_transform)
val_ds   = CropDataset(val_samples, val_transform)
test_ds  = CropDataset(te_smp,      val_transform)
print(f'Train: {len(train_ds):,}  Val: {len(val_ds):,}  Test: {len(test_ds):,}')

tr_labels     = [s[1] for s in tr_smp]
class_counts  = Counter(tr_labels)
w_per_class   = 1.0 / np.array([max(class_counts.get(i,1),1) for i in range(NUM_CLASSES)])
sample_w      = np.array([w_per_class[l] for l in tr_labels])
sampler       = WeightedRandomSampler(torch.FloatTensor(sample_w), len(train_ds), replacement=True)

_lkw = dict(num_workers=CFG['num_workers'], pin_memory=CFG['pin_memory'])
train_loader = DataLoader(train_ds, batch_size=CFG['batch_size'], sampler=sampler, drop_last=True,  **_lkw)
val_loader   = DataLoader(val_ds,   batch_size=CFG['batch_size']*2, shuffle=False, **_lkw)
test_loader  = DataLoader(test_ds,  batch_size=CFG['batch_size']*2, shuffle=False, **_lkw)
print(f'Batches -> train:{len(train_loader)}  val:{len(val_loader)}')


## Celda 7 — Modelo EfficientNet-B2

In [ ]:
import timm

class SpanishOCRModel(nn.Module):
    def __init__(self, num_classes, pretrained=True, drop_rate=0.3):
        super().__init__()
        self.backbone = timm.create_model(
            CFG['model_name'], pretrained=pretrained,
            num_classes=0, global_pool='avg', drop_rate=drop_rate)
        feat_dim = self.backbone.num_features
        self.head = nn.Sequential(
            nn.LayerNorm(feat_dim), nn.Dropout(drop_rate),
            nn.Linear(feat_dim, 512), nn.GELU(),
            nn.Dropout(drop_rate/2), nn.Linear(512, num_classes),
        )
    def forward(self, x): return self.head(self.backbone(x))

model = SpanishOCRModel(NUM_CLASSES, pretrained=CFG['pretrained'], drop_rate=CFG['drop_rate'])
if NUM_GPUS > 1:
    model = nn.DataParallel(model)
    print(f'DataParallel en {NUM_GPUS} GPUs')
model = model.to(DEVICE)
total_p = sum(p.numel() for p in model.parameters())
print(f'Params: {total_p:,}  |  Clases: {NUM_CLASSES}')


## Celda 8 — Optimizer, Scheduler, Loss

In [ ]:
import torch.optim as optim
from torch.amp import GradScaler

criterion = nn.CrossEntropyLoss(label_smoothing=CFG['label_smoothing']).to(DEVICE)

decay_p, no_decay_p = [], []
for name, p in model.named_parameters():
    if not p.requires_grad: continue
    (no_decay_p if any(k in name for k in ('bias','norm','bn')) else decay_p).append(p)

optimizer = optim.AdamW([
    {'params': decay_p,    'weight_decay': CFG['weight_decay']},
    {'params': no_decay_p, 'weight_decay': 0.0},
], lr=CFG['lr'], betas=(0.9,0.999), eps=1e-8)

def _wc(epoch):
    if epoch < CFG['warmup_epochs']: return (epoch+1)/CFG['warmup_epochs']
    prog = (epoch - CFG['warmup_epochs']) / max(1, CFG['epochs'] - CFG['warmup_epochs'])
    return 0.5*(1+np.cos(np.pi*prog))

scheduler = optim.lr_scheduler.LambdaLR(optimizer, _wc)
scaler    = GradScaler('cuda', enabled=CFG['use_amp'])
print(f'Optimizer/Scheduler/Loss listos | AMP={CFG["use_amp"]}')


## Celda 9 — MixUp / CutMix

In [ ]:
def mixup_batch(x, y):
    lam = np.random.beta(CFG['mixup_alpha'], CFG['mixup_alpha'])
    idx = torch.randperm(x.size(0), device=x.device)
    return lam*x+(1-lam)*x[idx], y, y[idx], lam

def cutmix_batch(x, y):
    lam = np.random.beta(CFG['cutmix_alpha'], CFG['cutmix_alpha'])
    idx = torch.randperm(x.size(0), device=x.device)
    B,C,H,W = x.shape
    cut_h = int(H*np.sqrt(1-lam)); cut_w = int(W*np.sqrt(1-lam))
    cx = np.random.randint(W); cy = np.random.randint(H)
    x1=max(0,cx-cut_w//2); x2=min(W,cx+cut_w//2)
    y1=max(0,cy-cut_h//2); y2=min(H,cy+cut_h//2)
    x = x.clone(); x[:,:,y1:y2,x1:x2] = x[idx,:,y1:y2,x1:x2]
    return x, y, y[idx], 1-((x2-x1)*(y2-y1)/(W*H))

def mix_loss(pred,ya,yb,lam): return lam*criterion(pred,ya)+(1-lam)*criterion(pred,yb)
print(f'MixUp alpha={CFG["mixup_alpha"]}  CutMix alpha={CFG["cutmix_alpha"]}')


## Celda 10 — Entrenamiento Clasificador + MLflow

In [ ]:
import mlflow
from torch.amp import autocast
from tqdm.notebook import tqdm

def train_one_epoch(net, loader, opt, scaler, epoch):
    net.train(); total_loss = correct = total = 0
    for imgs, labels in tqdm(loader, desc=f'Train E{epoch+1}', leave=False):
        imgs = imgs.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)
        r = np.random.rand()
        if r < 0.4:   imgs, ya, yb, lam = mixup_batch(imgs, labels);  use_mix=True
        elif r < 0.7: imgs, ya, yb, lam = cutmix_batch(imgs, labels); use_mix=True
        else: use_mix = False
        opt.zero_grad(set_to_none=True)
        with autocast('cuda', enabled=CFG['use_amp']):
            out  = net(imgs)
            loss = mix_loss(out,ya,yb,lam) if use_mix else criterion(out,labels)
        scaler.scale(loss).backward()
        scaler.unscale_(opt)
        nn.utils.clip_grad_norm_(net.parameters(), CFG['grad_clip'])
        scaler.step(opt); scaler.update()
        total_loss += loss.item()*imgs.size(0)
        if not use_mix:
            correct += (out.argmax(1)==labels).sum().item(); total += labels.size(0)
    return total_loss/len(loader.dataset), correct/max(total,1)

@torch.no_grad()
def _evaluate(net, loader, desc='Val'):
    net.eval(); total_loss = correct = top5 = total = 0
    for imgs, labels in tqdm(loader, desc=desc, leave=False):
        imgs = imgs.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)
        with autocast('cuda', enabled=CFG['use_amp']):
            out = net(imgs); loss = criterion(out,labels)
        total_loss += loss.item()*imgs.size(0)
        correct    += (out.argmax(1)==labels).sum().item()
        k = min(5,NUM_CLASSES)
        top5       += (out.topk(k,1)[1]==labels.unsqueeze(1)).any(1).sum().item()
        total += labels.size(0)
    return total_loss/len(loader.dataset), correct/total, top5/total

def _unwrap(m):
    while hasattr(m,'module') and isinstance(m.module,nn.Module): m=m.module
    return m

_net  = _unwrap(model).to(DEVICE)
best_val_acc = 0.0
best_ckpt    = str(CKPT_DIR / 'best_classifier.pt')
history      = {k:[] for k in ('train_loss','val_loss','train_acc','val_acc','val_top5','lr')}
patience_cnt = 0

mlflow.set_tracking_uri(CFG['mlflow_uri'])
mlflow.set_experiment(CFG['mlflow_experiment'])

print("=== Entrenamiento Clasificador ===\n")
with mlflow.start_run(run_name='classifier_efficientnet_b2') as clf_run:
    mlflow.log_params({
        'model':CFG['model_name'], 'epochs':CFG['epochs'],
        'batch_size':CFG['batch_size'], 'lr':CFG['lr'],
        'img_size':CFG['img_size'], 'num_classes':NUM_CLASSES,
        'amp':CFG['use_amp'], 'mixup':CFG['mixup_alpha'],
        'cutmix':CFG['cutmix_alpha'], 'label_smoothing':CFG['label_smoothing'],
        'train_samples':len(train_ds), 'val_samples':len(val_ds), 'n_gpus':NUM_GPUS,
    })
    for epoch in range(CFG['epochs']):
        t0 = time.time()
        tr_loss, tr_acc        = train_one_epoch(_net, train_loader, optimizer, scaler, epoch)
        va_loss, va_acc, va_t5 = _evaluate(_net, val_loader, 'Val')
        scheduler.step()
        lr_now  = optimizer.param_groups[0]['lr']
        elapsed = time.time()-t0
        for k,v in [('train_loss',tr_loss),('val_loss',va_loss),('train_acc',tr_acc),
                    ('val_acc',va_acc),('val_top5',va_t5),('lr',lr_now)]:
            history[k].append(v)
        mlflow.log_metrics({
            'train/loss':tr_loss, 'train/acc':tr_acc,
            'val/loss':va_loss, 'val/acc_top1':va_acc, 'val/acc_top5':va_t5,
            'lr':lr_now, 'epoch_sec':elapsed,
        }, step=epoch)
        print(f'E{epoch+1:3d}/{CFG["epochs"]} | '
              f'tr={tr_loss:.4f}/{tr_acc:.4f} | '
              f'va={va_loss:.4f}/{va_acc:.4f} top5={va_t5:.4f} | '
              f'lr={lr_now:.2e} | {elapsed:.1f}s')
        if va_acc > best_val_acc:
            best_val_acc = va_acc; patience_cnt = 0
            torch.save({'epoch':epoch,'model_state':_net.state_dict(),
                        'best_val_acc':best_val_acc,'num_classes':NUM_CLASSES}, best_ckpt)
            print(f'  Checkpoint guardado (acc={best_val_acc:.4f})')
        else:
            patience_cnt += 1
            if CFG['patience'] > 0 and patience_cnt >= CFG['patience']:
                print(f'\n  Early stop ep {epoch+1}'); break
    mlflow.log_metrics({'final/best_val_acc': best_val_acc})
    mlflow.log_artifact(best_ckpt, artifact_path='classifier')
    mlflow.log_artifact(str(char_map_path), artifact_path='classifier')
    CLF_RUN_ID = clf_run.info.run_id

print(f'\nClasificador: best_val_acc={best_val_acc:.4f}  run_id={CLF_RUN_ID}')


## Celda 11 — Curvas de entrenamiento

In [ ]:
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1,3,figsize=(18,5))
ep = range(1, len(history['train_loss'])+1)
axes[0].plot(ep,history['train_loss'],label='Train'); axes[0].plot(ep,history['val_loss'],label='Val')
axes[0].set_title('Loss'); axes[0].legend(); axes[0].grid(True)
axes[1].plot(ep,history['train_acc'],label='Train')
axes[1].plot(ep,history['val_acc'],label='Val Top-1')
axes[1].plot(ep,history['val_top5'],label='Val Top-5',linestyle='--')
axes[1].set_title('Accuracy'); axes[1].legend(); axes[1].grid(True)
axes[2].plot(ep,history['lr']); axes[2].set_title('LR'); axes[2].set_yscale('log'); axes[2].grid(True)
plt.tight_layout()
plt.savefig(str(OUTPUT_DIR/'training_curves.png'), dpi=150)
plt.show()


## Celda 12 — Evaluacion final en test

In [ ]:
import torch.serialization
from sklearn.metrics import classification_report

torch.serialization.add_safe_globals([np.float64, np.int64, np.int32])
ckpt_data  = torch.load(best_ckpt, map_location=DEVICE, weights_only=False)
eval_model = SpanishOCRModel(NUM_CLASSES, pretrained=False)
eval_model.load_state_dict(ckpt_data['model_state'])
eval_model = eval_model.to(DEVICE).eval()
print(f'Modelo cargado | epoch={ckpt_data["epoch"]+1} | acc={ckpt_data["best_val_acc"]:.4f}')

te_loader = DataLoader(test_ds, batch_size=256, num_workers=2, pin_memory=True)
te_loss, te_acc, te_top5 = _evaluate(eval_model, te_loader, 'Test')
print(f'Test => Loss:{te_loss:.4f}  Top-1:{te_acc:.4f} ({te_acc*100:.2f}%)  Top-5:{te_top5:.4f}')

all_preds, all_true = [], []
with torch.no_grad():
    for imgs,lbls in tqdm(te_loader, desc='Report', leave=False):
        preds = eval_model(imgs.to(DEVICE)).argmax(1).cpu().numpy()
        all_preds.extend(preds.tolist()); all_true.extend(lbls.numpy().tolist())

present = sorted(set(all_true))
names   = [IDX2CHAR.get(i,f'idx{i}') for i in present]
print(classification_report(all_true, all_preds, labels=present, target_names=names,
                             zero_division=0, digits=3)[:3000])


## Celda 13 — Exportar Clasificador a ONNX + INT8

In [ ]:
import onnx, onnxruntime as ort
from onnxsim import simplify

onnx_fp32 = str(ONNX_DIR / 'classifier_fp32.onnx')
onnx_sim  = str(ONNX_DIR / 'classifier_simplified.onnx')
onnx_int8 = str(ONNX_DIR / 'classifier_int8.onnx')

eval_model.eval().cpu()
dummy = torch.randn(1, 3, CFG['img_size'], CFG['img_size'])

print('Exportando ONNX FP32 ...')
torch.onnx.export(
    eval_model, dummy, onnx_fp32,
    export_params=True, opset_version=17, do_constant_folding=True, dynamo=False,
    input_names=['image'], output_names=['logits'],
    dynamic_axes={'image':{0:'batch'},'logits':{0:'batch'}})
print(f'  FP32 -> {os.path.getsize(onnx_fp32)/1e6:.1f} MB')

try:
    m_sim, ok = simplify(onnx.load(onnx_fp32))
    if ok: onnx.save(m_sim, onnx_sim); print(f'  Simplified -> {os.path.getsize(onnx_sim)/1e6:.1f} MB')
    else:  shutil.copy(onnx_fp32, onnx_sim)
except Exception as e:
    print(f'  Simplify fallo ({e}); copiando FP32'); shutil.copy(onnx_fp32, onnx_sim)

try:
    from onnxruntime.quantization import quantize_dynamic, QuantType
    quantize_dynamic(onnx_sim, onnx_int8, weight_type=QuantType.QInt8)
    print(f'  INT8 -> {os.path.getsize(onnx_int8)/1e6:.1f} MB')
except Exception as e:
    print(f'  INT8 fallo: {e}')

with torch.no_grad():
    pt_out = eval_model(dummy).numpy()
sess = ort.InferenceSession(onnx_sim, providers=['CPUExecutionProvider'])
ort_out = sess.run(None, {'image': dummy.numpy()})[0]
print(f'  Max diff PyTorch vs ONNX: {np.abs(pt_out-ort_out).max():.6f}')

with mlflow.start_run(run_id=CLF_RUN_ID):
    for p in [onnx_fp32, onnx_sim, onnx_int8]:
        if os.path.exists(p): mlflow.log_artifact(p, artifact_path='onnx')
    mlflow.log_metrics({'test/top1':te_acc,'test/top5':te_top5})
print('Artefactos ONNX clasificador en MLflow')


## Celda 14 — Entrenar Detector YOLOv8n + MLflow

In [ ]:
import yaml

# 1. Definir la ruta del nuevo YAML que vamos a crear
KAGGE_YAML_PATH = '/kaggle/working/dataset_kaggle.yaml'

# 2. Configurar el contenido con las rutas de Kaggle que detectamos antes
# Usamos las variables que definimos en la celda anterior (_DATASET_ROOT)
data_config = {
    'path': str(_DATASET_ROOT),      # La raíz del dataset en Kaggle
    'train': 'images/train',         # Ruta relativa a la raíz
    'val': 'images/val',             # Ruta relativa a la raíz
    'nc': 62,                        # Número de clases (0-9, A-Z, a-z)
    'names': [
        '0','1','2','3','4','5','6','7','8','9',
        'A','B','C','D','E','F','G','H','I','J','K','L','M','N','O','P','Q','R','S','T','U','V','W','X','Y','Z',
        'a','b','c','d','e','f','g','h','i','j','k','l','m','n','o','p','q','r','s','t','u','v','w','x','y','z'
    ]
}

# 3. Guardar el archivo en la carpeta donde sí tenemos permiso de escritura
with open(KAGGE_YAML_PATH, 'w') as f:
    yaml.dump(data_config, f, default_flow_style=False)

# 4. ACTUALIZAR la variable que usa YOLO
DATASET_YAML = KAGGE_YAML_PATH

print(f"✅ Nuevo archivo YAML creado en: {DATASET_YAML}")
print(f"📍 Apuntando a imágenes en: {data_config['path']}")

In [ ]:
# =============================================================
# CELDA OPTIMIZADA: Detector YOLO — Fix velocidad
# Problema: dataset en /kaggle/input/ (NFS lento) + cache RAM insuficiente
# Solución: copiar a /kaggle/working/ (SSD local) + epochs reducidas
# =============================================================
import shutil, yaml
from pathlib import Path
from ultralytics import YOLO

# ── 1. Copiar dataset a SSD local (/kaggle/working/) ──────────
FAST_DS_DIR = Path('/kaggle/working/yolo_dataset_local')

if not FAST_DS_DIR.exists():
    print("Copiando dataset a SSD local (solo la primera vez, ~2-3 min)...")
    shutil.copytree(str(_DATASET_ROOT), str(FAST_DS_DIR))
    print(f"  Copiado: {sum(1 for _ in FAST_DS_DIR.rglob('*.jpg')):,} imágenes")
else:
    print(f"Dataset local ya existe: {FAST_DS_DIR}")

# ── 2. Crear dataset.yaml apuntando a SSD local ───────────────
with open(DATASET_YAML, 'r') as f:
    ds_yaml = yaml.safe_load(f)

ds_yaml['path'] = str(FAST_DS_DIR)
ds_yaml['train'] = 'images/train'
ds_yaml['val']   = 'images/val'

DATASET_YAML_LOCAL = '/kaggle/working/dataset_local.yaml'
with open(DATASET_YAML_LOCAL, 'w') as f:
    yaml.dump(ds_yaml, f)
print(f"YAML local: {DATASET_YAML_LOCAL}")

# ── 3. Config YOLO optimizado ─────────────────────────────────
# El modelo ya alcanza mAP50=0.993 en época 1 con este dataset.
# Con 15 épocas y patience=5 termina en ~40 min en lugar de 20+ horas.
YOLO_CFG_FAST = dict(
    model_variant  = 'yolov8n.pt',
    epochs         = 10,          # mAP converge en 2-3 épocas; 15 es más que suficiente
    batch          = 64,          # 2× T4 = 2×14GB VRAM; batch 64/GPU es seguro con yolov8n
    device         = '0,1',
    workers        = 4,           # 4 workers es óptimo para SSD local en Kaggle
    img_size       = 640,
    lr0            = 0.01,
    lrf            = 0.01,
    momentum       = 0.937,
    weight_decay   = 5e-4,
    warmup_epochs  = 3.0,         # warmup más corto para pocas épocas
    amp            = True,
    cache          = False,       # SSD local es suficientemente rápido; RAM insuficiente
    patience       = 5,           # early stop rápido: mAP ya es 0.993 desde época 1
    project        = str(YOLO_DIR),
    name           = 'char_detector_t4_fast',
    exist_ok       = True,
    pretrained     = True,
)

n_train = sum(1 for _ in FAST_DS_DIR.glob('images/train/*.jpg'))
n_val   = sum(1 for _ in FAST_DS_DIR.glob('images/val/*.jpg'))
print(f"\nConfig entrenamiento:")
print(f"  Train/Val  : {n_train:,} / {n_val:,} imágenes")
print(f"  Épocas     : {YOLO_CFG_FAST['epochs']} (patience={YOLO_CFG_FAST['patience']})")
print(f"  Batch/GPU  : {YOLO_CFG_FAST['batch']}")
print(f"  Cache      : {YOLO_CFG_FAST['cache']} (SSD local = no necesario)")
print(f"  Tiempo est : ~{(n_train // YOLO_CFG_FAST['batch']) * YOLO_CFG_FAST['epochs'] // 120:.0f} min")

# ── 4. MLflow ─────────────────────────────────────────────────
mlflow.set_tracking_uri(CFG['mlflow_uri'])
mlflow.set_experiment(CFG['mlflow_experiment'])

class _MLflowYOLOCB:
    def __init__(self): self.run_id = None; self._ctx = None

    def on_train_start(self, trainer):
        run = mlflow.start_run(run_name='detector_yolov8n_fast')
        self.run_id = run.info.run_id; self._ctx = run
        mlflow.log_params({
            'model'       : YOLO_CFG_FAST['model_variant'],
            'epochs'      : YOLO_CFG_FAST['epochs'],
            'batch'       : YOLO_CFG_FAST['batch'],
            'device'      : YOLO_CFG_FAST['device'],
            'img_size'    : YOLO_CFG_FAST['img_size'],
            'cache'       : str(YOLO_CFG_FAST['cache']),
            'dataset_src' : 'ssd_local_copy',
            'n_train'     : n_train,
            'n_val'       : n_val,
        })

    def on_fit_epoch_end(self, trainer):
        if not self.run_id: return
        epoch   = trainer.epoch
        metrics = trainer.metrics or {}
        losses  = trainer.loss_items
        log = {}
        if losses is not None and len(losses) >= 3:
            log['train/box_loss'] = float(losses[0])
            log['train/cls_loss'] = float(losses[1])
            log['train/dfl_loss'] = float(losses[2])
        for uk, mk in [
            ('metrics/mAP50(B)',    'val/mAP50'),
            ('metrics/mAP50-95(B)', 'val/mAP50_95'),
            ('metrics/precision(B)','val/precision'),
            ('metrics/recall(B)',   'val/recall'),
        ]:
            if uk in metrics: log[mk] = float(metrics[uk])
        if log: mlflow.log_metrics(log, step=epoch)

    def on_train_end(self, trainer):
        if self._ctx: mlflow.end_run()

yolo_cb   = _MLflowYOLOCB()
det_model = YOLO(YOLO_CFG_FAST['model_variant'])
det_model.add_callback('on_train_start',   yolo_cb.on_train_start)
det_model.add_callback('on_fit_epoch_end', yolo_cb.on_fit_epoch_end)
det_model.add_callback('on_train_end',     yolo_cb.on_train_end)

print("\n=== Iniciando entrenamiento optimizado ===\n")
yolo_results = det_model.train(
    data          = DATASET_YAML_LOCAL,
    epochs        = YOLO_CFG_FAST['epochs'],
    batch         = YOLO_CFG_FAST['batch'],
    imgsz         = YOLO_CFG_FAST['img_size'],
    device        = YOLO_CFG_FAST['device'],
    workers       = YOLO_CFG_FAST['workers'],
    lr0           = YOLO_CFG_FAST['lr0'],
    lrf           = YOLO_CFG_FAST['lrf'],
    momentum      = YOLO_CFG_FAST['momentum'],
    weight_decay  = YOLO_CFG_FAST['weight_decay'],
    warmup_epochs = YOLO_CFG_FAST['warmup_epochs'],
    amp           = YOLO_CFG_FAST['amp'],
    cache         = YOLO_CFG_FAST['cache'],
    patience      = YOLO_CFG_FAST['patience'],
    project       = YOLO_CFG_FAST['project'],
    name          = YOLO_CFG_FAST['name'],
    exist_ok      = YOLO_CFG_FAST['exist_ok'],
    pretrained    = YOLO_CFG_FAST['pretrained'],
)

print('\nYOLO completado')
try:
    res = yolo_results.results_dict
    print(f'  mAP50    : {res.get("metrics/mAP50(B)",    0):.4f}')
    print(f'  mAP50-95 : {res.get("metrics/mAP50-95(B)", 0):.4f}')
    print(f'  Precision: {res.get("metrics/precision(B)",0):.4f}')
    print(f'  Recall   : {res.get("metrics/recall(B)",   0):.4f}')
except: pass

## Celda 15 — Exportar Detector a ONNX

In [ ]:
import os
from pathlib import Path
from ultralytics import YOLO

# --- RUTA DIRECTA PROPORCIONADA ---
yolo_best_pt = "/kaggle/working/runs/detect/char_detector_t4_fast/weights/best.pt"

yolo_onnx_path = None

if Path(yolo_best_pt).exists():
    print(f'✅ Archivo encontrado, iniciando exportación: {yolo_best_pt}')
    
    # Cargar el modelo desde la ruta directa
    det_ex = YOLO(yolo_best_pt)
    
    # Exportar a ONNX
    # Usamos imgsz de YOLO_CFG si existe, de lo contrario 640 por defecto
    img_size = YOLO_CFG.get('img_size', 640)
    
    print(f'  Exportando a ONNX (imgsz={img_size})...')
    export_path = det_ex.export(
        format='onnx', 
        imgsz=img_size,
        opset=17, 
        dynamic=False, 
        simplify=True
    )
    
    # El método export devuelve la ruta del archivo generado
    yolo_onnx_path = str(export_path)
    
    if Path(yolo_onnx_path).exists():
        size_mb = os.path.getsize(yolo_onnx_path) / 1e6
        print(f'  🚀 ONNX generado con éxito: {yolo_onnx_path} ({size_mb:.2f} MB)')
    else:
        print('  ❌ Error: El proceso de exportación no generó el archivo esperado.')
        yolo_onnx_path = None
else:
    print(f'❌ ERROR: No se encontró el archivo en la ruta: {yolo_best_pt}')
    print('Verifica si el nombre de la carpeta "char_detector_t4_fast" es correcto.')

# --- LOG EN MLFLOW ---
# Solo si existe el run_id y el archivo ONNX
if 'yolo_cb' in globals() and yolo_cb.run_id and yolo_onnx_path and Path(yolo_onnx_path).exists():
    import mlflow
    with mlflow.start_run(run_id=yolo_cb.run_id):
        print(f'📦 Registrando artefactos en MLflow (Run: {yolo_cb.run_id})...')
        mlflow.log_artifact(yolo_onnx_path, artifact_path='onnx')
        mlflow.log_artifact(yolo_best_pt, artifact_path='weights')
    print('✅ Artefactos YOLO registrados en MLflow.')

## Celda 16 — Copiar artefactos finales

In [ ]:
FINAL_DIR = OUTPUT_DIR / 'final_models'
FINAL_DIR.mkdir(exist_ok=True)

def _cp(src, name):
    src = Path(src) if src else None
    if src and src.exists():
        dst = FINAL_DIR/name; shutil.copy2(src, dst)
        print(f'  {name}  ({dst.stat().st_size/1e6:.1f} MB)'); return str(dst)
    print(f'  !! {name}: no encontrado'); return None

print('=== Copiando modelos finales ===')
_cp(onnx_sim,          'mobilenet_classifier.onnx')
_cp(onnx_int8,         'mobilenet_classifier_int8.onnx')
_cp(yolo_onnx_path,    'yolo_detector.onnx')
_cp(char_map_path,     'char_map.json')
_cp(best_ckpt,         'best_classifier.pt')
if yolo_best_pt: _cp(yolo_best_pt, 'best_detector.pt')

print(f'\nContenido de {FINAL_DIR}:')
for f in sorted(FINAL_DIR.iterdir()):
    print(f'  {f.name:45s}  {f.stat().st_size/1e6:.2f} MB')


## Celda 17 — Paquete ZIP + Verificacion Final

In [ ]:
import zipfile

metadata = {
    'project':'Tutor Inteligente OCR Espanol','version':'2.0.0','date':'2026-03',
    'classifier':{'model':CFG['model_name'],'num_classes':NUM_CLASSES,
                  'img_size':CFG['img_size'],'best_val_acc':round(best_val_acc,4),
                  'input':'image','output':'logits',
                  'preprocessing':{'resize':[CFG['img_size'],CFG['img_size']],
                                   'normalize':{'mean':[0.485,0.456,0.406],'std':[0.229,0.224,0.225]}}},
    'detector':{'model':YOLO_CFG['model_variant'],'img_size':YOLO_CFG['img_size'],'nc':1,'class':'trazo'},
    'mlflow_uri':CFG['mlflow_uri'],'onnx_opset':17,
    'recommended_clf':'mobilenet_classifier_int8.onnx',
    'recommended_det':'yolo_detector.onnx',
}
with open(FINAL_DIR/'model_metadata.json','w',encoding='utf-8') as f:
    json.dump(metadata,f,ensure_ascii=False,indent=2)

zip_path = str(OUTPUT_DIR/'tutor_inteligente_models.zip')
with zipfile.ZipFile(zip_path,'w',zipfile.ZIP_DEFLATED) as zf:
    for fp in sorted(FINAL_DIR.iterdir()): zf.write(fp,fp.name)
    curves = OUTPUT_DIR/'training_curves.png'
    if curves.exists(): zf.write(curves,'training_curves.png')

print(f'ZIP: {zip_path}  ({os.path.getsize(zip_path)/1e6:.1f} MB)')
with zipfile.ZipFile(zip_path) as zf:
    for i in zf.infolist():
        print(f'  {i.filename:50s}  {i.file_size/1e6:.2f} MB')

# ---- Verificacion clasificador ----
print('\n=== Verificacion ONNX Clasificador ===')
clf_onnx_f = str(FINAL_DIR/'mobilenet_classifier.onnx')
if Path(clf_onnx_f).exists():
    s = ort.InferenceSession(clf_onnx_f, providers=['CPUExecutionProvider'])
    ti = np.random.randn(1,3,CFG['img_size'],CFG['img_size']).astype(np.float32)
    out = s.run(None,{'image':ti})[0]
    pid = int(np.argmax(out[0]))
    print(f'  Shape out: {out.shape}  |  pred_idx={pid}  char={repr(IDX2CHAR.get(pid,"?"))}')
    print('  OK')

# ---- Verificacion detector ----
print('\n=== Verificacion ONNX Detector ===')
det_onnx_f = str(FINAL_DIR/'yolo_detector.onnx')
if Path(det_onnx_f).exists():
    sd = ort.InferenceSession(det_onnx_f, providers=['CPUExecutionProvider'])
    inp_n = sd.get_inputs()[0].name
    td = np.random.randn(1,3,640,640).astype(np.float32)
    od = sd.run(None,{inp_n:td})[0]
    print(f'  Input: {td.shape}  Output: {od.shape}')
    print('  OK')
else:
    print('  yolo_detector.onnx no encontrado')

print('\n' + '='*55)
print('PIPELINE COMPLETADO')
print(f'  Clasificador best acc : {best_val_acc*100:.2f}%')
print(f'  MLflow tracking      : {CFG["mlflow_uri"]}')
print(f'  ZIP final            : {zip_path}')
print('='*55)


In [ ]:
from IPython.display import FileLink

# Genera el enlace de descarga
FileLink(r'tutor_inteligente_models.zip')